# 01 - Data Audit
Scan all videos in the Google Drive folder and generate a metadata inventory.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/LightningPoseTrack.git"  # TODO: change this
GIT_BRANCH = "main"

# Path to the shared Drive folder containing session folders (added as shortcut to My Drive)
DRIVE_RAW_VIDEOS = "/content/drive/My Drive/PigBehavior/raw_videos"  # TODO: update to your path

# Where to save outputs on Drive
DRIVE_OUTPUT = "/content/drive/My Drive/PigBehavior/reports"

# Google Drive folder ID (alternative: use this to mount directly)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet opencv-python pandas numpy pyarrow

In [ ]:
from pathlib import Path
from src.io.video_inventory import scan_videos

# Check what's in the raw videos directory
import subprocess
result = subprocess.run(["ls", "-la", DRIVE_RAW_VIDEOS], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} video files across {df['session'].nunique()} sessions")
df.head(20)

In [ ]:
print("=== Summary ===")
print(f"Total videos: {len(df)}")
print(f"Total duration: {df['duration_min'].sum():.1f} min")
print(f"Sessions: {sorted(df['session'].unique())}")
print(f"Cameras: {sorted(df['camera'].unique())}")
print(f"\nPer session:")
print(df.groupby('session').agg(
    videos=('filename', 'count'),
    duration_min=('duration_min', 'sum'),
    cameras=('camera', lambda x: sorted(x.unique()))
))

In [ ]:
output_dir = Path(DRIVE_OUTPUT)
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "video_inventory.csv"
df.to_csv(csv_path, index=False)
print(f"Saved inventory to {csv_path}")